# ReLU Variants

---

## 1. Quick Recap — Standard ReLU

Before looking at variants, recall the standard ReLU:

$$\text{ReLU}(z) = \max(0, z) = \begin{cases} z & z > 0 \\ 0 & z \leq 0 \end{cases}$$

$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

ReLU solved the vanishing gradient problem for positive inputs. But it introduced a new one.

---

## 2. The Dying ReLU Problem

### What happens

When a neuron receives a **negative input**, ReLU outputs exactly **0**. During backpropagation, the gradient is also 0:

$$\frac{\partial L}{\partial w} = \frac{\partial L}{\partial z} \cdot \underbrace{\text{ReLU}'(z)}_{= \ 0} = 0$$

A weight update of zero means the weight **never changes**:

$$w_{\text{new}} = w_{\text{old}} - \eta \cdot 0 = w_{\text{old}}$$

This neuron is now **dead** — it will output 0 for every input forever, because its weights can never recover.

### Why it happens

- A large learning rate causes weights to overshoot into large negative values
- Once $z = w \cdot x + b$ becomes consistently negative for all training examples, the neuron is permanently stuck
- In the worst case, **entire layers** can die, effectively making the network shallower

### How to detect it

Monitor the fraction of neurons outputting 0 during training. If a large portion stays at 0 across many epochs, dying ReLU is likely.

### Solutions

| Solution | How it helps |
|----------|-------------|
| **Use a smaller learning rate** | Prevents weights from overshooting into large negative values |
| **Initialize biases to a small positive value** | Keeps $z = wx + b$ positive at the start of training |
| **Use a ReLU variant** | Allows a non-zero gradient even for negative inputs (see below) |

---

## 3. Leaky ReLU

Instead of outputting a flat 0 for negative inputs, Leaky ReLU allows a **small, fixed negative slope**:

$$\text{Leaky ReLU}(z) = \max(0.01z,\ z) = \begin{cases} z & z > 0 \\ 0.01z & z \leq 0 \end{cases}$$

$$\text{Leaky ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0.01 & z \leq 0 \end{cases}$$

The derivative for negative inputs is **0.01** instead of 0, so the gradient never fully vanishes — there is always some signal flowing backward.

### Example

For $z = -5$:

$$\text{Standard ReLU}(-5) = 0 \quad \rightarrow \quad \text{gradient} = 0 \quad \text{(dead)}$$

$$\text{Leaky ReLU}(-5) = 0.01 \times (-5) = -0.05 \quad \rightarrow \quad \text{gradient} = 0.01 \quad \text{(alive)}$$

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Fixes the dying ReLU problem | The slope $0.01$ is a hyperparameter — may not be optimal for all tasks |
| Simple to implement | Negative outputs may not always be desirable |
| Consistent gradient flow for all inputs | Small gradient for negatives still slows learning in those regions |

---

## 4. Parametric ReLU (PReLU)

Leaky ReLU fixes the negative slope at $0.01$. **PReLU makes that slope a learnable parameter** $\alpha$, trained alongside the weights via backpropagation:

$$\text{PReLU}(z) = \begin{cases} z & z > 0 \\ \alpha z & z \leq 0 \end{cases}, \qquad \alpha \in \mathbb{R},\ \alpha > 0$$

$$\text{PReLU}'(z) = \begin{cases} 1 & z > 0 \\ \alpha & z \leq 0 \end{cases}$$

The gradient with respect to $\alpha$ (so it can be learned):

$$\frac{\partial L}{\partial \alpha} = \frac{\partial L}{\partial z} \cdot z \quad \text{(for } z \leq 0\text{)}$$

### Example

If the network learns $\alpha = 0.25$ for a particular layer, then for $z = -4$:

$$\text{PReLU}(-4) = 0.25 \times (-4) = -1.0 \quad \rightarrow \quad \text{gradient} = 0.25$$

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| $\alpha$ adapts to the data — more flexible than Leaky ReLU | Adds extra parameters to learn |
| Can outperform Leaky ReLU when data has complex patterns | Risk of overfitting on small datasets |
| Different layers can learn different $\alpha$ values | Slightly more complex to implement |

---

## 5. Exponential Linear Unit (ELU)

ELU uses an **exponential curve** for negative inputs instead of a linear slope:

$$\text{ELU}(z) = \begin{cases} z & z > 0 \\ \alpha(e^z - 1) & z \leq 0 \end{cases}, \qquad \alpha > 0 \text{ (typically } \alpha = 1\text{)}$$

$$\text{ELU}'(z) = \begin{cases} 1 & z > 0 \\ \alpha e^z = \text{ELU}(z) + \alpha & z \leq 0 \end{cases}$$

### Example

For $z = -2$, $\alpha = 1$:

$$\text{ELU}(-2) = 1 \times (e^{-2} - 1) = 0.135 - 1 = -0.865$$

$$\text{ELU}'(-2) = e^{-2} = 0.135$$

The output for negative inputs approaches $-\alpha$ (not $-\infty$), which **centers the activations near zero** — similar to the benefit of batch normalization.

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| Outputs push mean activation toward zero (faster convergence) | Involves $e^z$ — more computationally expensive than ReLU/Leaky |
| Smooth and differentiable everywhere (no kink at 0) | $\alpha$ is a hyperparameter to tune |
| Stronger gradient signal for negative inputs than Leaky ReLU | Outputs can be negative — not always desirable |

---

## 6. Scaled Exponential Linear Unit (SELU)

SELU is a **self-normalizing** variant of ELU. It uses carefully chosen fixed constants $\alpha$ and $\lambda$ so that activations automatically maintain **mean ≈ 0 and variance ≈ 1** across layers — without needing batch normalization.

$$\text{SELU}(z) = \lambda \begin{cases} z & z > 0 \\ \alpha(e^z - 1) & z \leq 0 \end{cases}$$

$$\text{SELU}'(z) = \lambda \begin{cases} 1 & z > 0 \\ \alpha e^z & z \leq 0 \end{cases}$$

The fixed constants (derived mathematically to ensure self-normalization):

$$\alpha \approx 1.6733, \qquad \lambda \approx 1.0507$$

### Example

For $z = -1$:

$$\text{SELU}(-1) = 1.0507 \times 1.6733 \times (e^{-1} - 1) = 1.0507 \times 1.6733 \times (-0.632) \approx -1.111$$

For $z = 2$:

$$\text{SELU}(2) = 1.0507 \times 2 = 2.101$$

| ✅ Advantages | ❌ Disadvantages |
|---|---|
| **Self-normalizing** — no need for batch normalization | Only works well with specific weight initialization (LeCun normal) |
| Activations stay stable in mean and variance across deep layers | Requires fully connected layers — less effective with CNNs |
| Strong theoretical guarantees for deep networks | Less commonly used in practice compared to ReLU + BatchNorm |

---

## 7. Full Comparison

| Activation | Formula (negative side) | Derivative (negative side) | Dying neuron? | Learnable? |
|---|---|---|---|---|
| **ReLU** | $0$ | $0$ | ✅ Yes | ❌ |
| **Leaky ReLU** | $0.01z$ | $0.01$ | ❌ No | ❌ |
| **PReLU** | $\alpha z$ | $\alpha$ | ❌ No | ✅ Yes |
| **ELU** | $\alpha(e^z - 1)$ | $\alpha e^z$ | ❌ No | ❌ |
| **SELU** | $\lambda\alpha(e^z - 1)$ | $\lambda\alpha e^z$ | ❌ No | ❌ |

```
Output
  ▲
  │         ╱  ← ReLU / Leaky / PReLU / ELU / SELU all the same for z > 0
  │       ╱
  │     ╱
──┼───╱──────────────────► z
  │ ╱╲
  │╱   ╲_ _ _  ← ReLU: flat 0
  │  ╲          ← Leaky/PReLU: small slope (−0.01z or −αz)
  │    ╲_ _ _  ← ELU/SELU: exponential curve approaching −α
```

---

## 8. When to Use Which

| Scenario | Recommended |
|----------|------------|
| Default starting point | **ReLU** |
| Dead neurons appearing during training | **Leaky ReLU** |
| Want the network to learn the best negative slope | **PReLU** |
| Want zero-centered outputs, smooth curve | **ELU** |
| Very deep fully connected network, want to skip batch norm | **SELU** |

---

## 9. Key Takeaways

- **Dying ReLU** happens because the gradient is exactly 0 for all negative inputs — once a neuron dies, it cannot recover.
- All variants fix this by ensuring a **non-zero gradient for negative inputs**, either with a fixed slope (Leaky), a learned slope (PReLU), or an exponential curve (ELU, SELU).
- **SELU** is the most powerful theoretically but the most demanding in terms of requirements (weight init, architecture).
- In practice, **ReLU** remains the default. Switch to Leaky ReLU or ELU if dying neurons become a problem.